In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, average_precision_score
import shap
import pickle
import dagshub
import mlflow
import dagshub.auth

In [ ]:
%pip install mlflow dagshub tf-keras --ignore-installed blinker

In [ ]:
%pip install tensorflow

In [ ]:
%pip install shap

In [ ]:
DAGSHUB_REPO_OWNER = "dagshub_name"
DAGSHUB_REPO_NAME = "dagshub_name"
DAGSHUB_TOKEN = "dagshub_token"

In [ ]:
my_token =DAGSHUB_TOKEN

dagshub.auth.add_app_token(my_token)

In [ ]:
dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME , mlflow=True)


Initialized MLflow to track repo "oke03940/Credit-Scout-Fraud-Detection"

Repository oke03940/Credit-Scout-Fraud-Detection initialized!

In [ ]:
mlflow.set_experiment("Credit-Scout-Fraud-Detection")

2026/01/06 21:38:05 INFO mlflow.tracking.fluent: Experiment with name 'Credit-Scout-Fraud-Detection' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/91f6e89c55ec4b45bc6cdfcf7829d562', creation_time=1767735485931, experiment_id='0', last_update_time=1767735485931, lifecycle_stage='active', name='Credit-Scout-Fraud-Detection', tags={}>

In [ ]:
BATCH_SIZE = 64
EPOCHS = 20

In [ ]:
X_train = np.load('X_train.npy')
X_test = np.load('X_test.npy')
y_train = np.load('y_train.npy')
y_test = np.load('y_test.npy')


In [ ]:
with mlflow.start_run():
    # --- A. LOG PARAMETERS ---
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("model_type", "Bi-LSTM")
    mlflow.log_param("features", len(feature_names))


🏃 View run suave-pig-453 at: https://dagshub.com/oke03940/Credit-Scout-Fraud-Detection.mlflow/#/experiments/0/runs/8111b58f7cfe4470be624f29fced6b4f
🧪 View experiment at: https://dagshub.com/oke03940/Credit-Scout-Fraud-Detection.mlflow/#/experiments/0


In [ ]:
model = Sequential([
  Bidirectional(LSTM(64, return_sequences=True), input_shape=(X_train.shape[1], X_train.shape[2])),
  Dropout(0.3),
  Bidirectional(LSTM(32)),
  Dropout(0.3),
  Dense(16, activation='relu'),
  Dense(1, activation='sigmoid')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['AUC'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
mlflow.tensorflow.autolog()

weights = {0: 1, 1: 2.0}
callbacks = [EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]

history = model.fit(
  X_train, y_train,
  validation_split=0.2,
  epochs=EPOCHS,
  batch_size=BATCH_SIZE,
  class_weight=weights,
  callbacks=callbacks,
  verbose=1
)


2026/01/06 21:44:40 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '1dc2aafa9c6141a2b8aaea0a7701e5d4', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current tensorflow workflow


Epoch 1/20
490/493 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - AUC: 0.8700 - loss: 0.4861

493/493 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - AUC: 0.9141 - loss: 0.3811 - val_AUC: 0.9396 - val_loss: 0.2199
Epoch 2/20
482/493 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - AUC: 0.9386 - loss: 0.3008

493/493 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - AUC: 0.9376 - loss: 0.3035 - val_AUC: 0.9449 - val_loss: 0.2145
Epoch 3/20
488/493 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - AUC: 0.9388 - loss: 0.2968

493/493 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - AUC: 0.9413 - loss: 0.2911 - val_AUC: 0.9453 - val_loss: 0.2075
Epoch 4/20
492/493 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - AUC: 0.9412 - loss: 0.2879

...

493/493 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - AUC: 0.9435 - loss: 0.2842 - val_AUC: 0.9482 - val_loss: 0.1992
Epoch 9/20
489/493 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - AUC: 0.9462 - loss: 0.2787

493/493 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - AUC: 0.9451 - loss: 0.2802 - val_AUC: 0.9480 - val_loss: 0.1988
Epoch 10/20
491/493 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - AUC: 0.9438 - loss: 0.2823

493/493 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - AUC: 0.9438 - loss: 0.2809 - val_AUC: 0.9476 - val_loss: 0.1966
Epoch 11/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - AUC: 0.9462 - loss: 0.2760 - val_AUC: 0.9488 - val_loss: 0.2051
Epoch 12/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - AUC: 0.9447 - loss: 0.2775 - val_AUC: 0.9490 - val_loss: 0.1992
Epoch 13/20
493/493 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - AUC: 0.9456 - loss: 0.2780 - val_AUC: 0.9495 - val_loss: 0.1975
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 356ms/step


2026/01/06 21:45:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run gregarious-snake-281 at: https://dagshub.com/oke03940/Credit-Scout-Fraud-Detection.mlflow/#/experiments/0/runs/1dc2aafa9c6141a2b8aaea0a7701e5d4
🧪 View experiment at: https://dagshub.com/oke03940/Credit-Scout-Fraud-Detection.mlflow/#/experiments/0


In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

308/308 ━━━━━━━━━━━━━━━━━━━━ 1s 841us/step


In [ ]:
auprc = average_precision_score(y_test, y_pred_prob)
print(f"AUPRC: {auprc:.4f}")

# Log manual metrics that autolog might miss
mlflow.log_metric("test_auprc", auprc)

AUPRC: 0.7224


In [ ]:
print("Generating SHAP Explanations...")
# Background sample for SHAP
background_sample = X_train[np.random.choice(X_train.shape[0], 200, replace=False)]

Generating SHAP Explanations...


In [ ]:
shap_data = {
        'background_sample': background_sample,
        'feature_names': feature_names
    }

with open("shap_metadata.pkl", "wb") as f:
        pickle.dump(shap_data, f)


In [ ]:
mlflow.log_artifact("shap_metadata.pkl")
print("SHAP metadata logged to MLflow.")

SHAP metadata logged to MLflow.
